# Combined Transcriptomics Cleaning
Processes 2019_UGA, 2020_UGA, and SDY2867 with a shared pipeline, then concatenates into a single parquet.

In [1]:
import sys
sys.path.insert(0, '../..')

import pandas as pd

from data_cleaning.utils import log_standard_scale, peek

DATA_PATH = "../../data"
CLEAN_DATA_PATH = "../../cleaned_data"

In [2]:
unique_genes_challenge = set(
    pd.read_csv(DATA_PATH + '/challenge_transcriptomics.tsv', sep='\t')['ensembl_gene_id'].unique()
)
print(f"Challenge genes: {len(unique_genes_challenge)}")

Challenge genes: 54902


## EDA

In [3]:
eda_2019 = pd.read_csv(DATA_PATH + '/train_transcriptomics_2019_UGA.tsv', sep='\t')
eda_2020 = pd.read_csv(DATA_PATH + '/train_transcriptomics_2020_UGA.tsv', sep='\t')
eda_sdy  = pd.read_csv(DATA_PATH + '/train_transcriptomics_SDY2867.tsv', sep='\t')

In [4]:
for name, df in [('2019_UGA', eda_2019), ('2020_UGA', eda_2020), ('SDY2867', eda_sdy)]:
    print(f"\n{name} — timepoint value counts:")
    print(df['timepoint'].value_counts().sort_index())
    print(df.groupby('timepoint')['tpm_count'].mean())


2019_UGA — timepoint value counts:
timepoint
0     9819425
3     6901257
7     6901257
28    6901257
Name: count, dtype: int64
timepoint
0      28.005713
3     242.006574
7     217.147233
28    191.529194
Name: tpm_count, dtype: float64

2020_UGA — timepoint value counts:
timepoint
0     2909616
28    2909616
Name: count, dtype: int64
timepoint
0     17.726096
28    17.726096
Name: tpm_count, dtype: float64

SDY2867 — timepoint value counts:
timepoint
-14    4007846
 0     3952944
 1     3952944
 7     3952944
 28    3952944
Name: count, dtype: int64
timepoint
-14    18.214273
 0     18.214273
 1     18.214273
 7     18.214273
 28    18.214273
Name: tpm_count, dtype: float64


In [5]:
for name, df in [('2019_UGA', eda_2019), ('2020_UGA', eda_2020), ('SDY2867', eda_sdy)]:
    print(f"\n{name} — average tpm_count per gene (first 10):")
    print(df.groupby('ensembl_gene_id')['tpm_count'].mean().head(10))


2019_UGA — average tpm_count per gene (first 10):
ensembl_gene_id
ENSG00000000003       3.273456
ENSG00000000005       0.001650
ENSG00000000419     100.951028
ENSG00000000457     148.218532
ENSG00000000460      24.120888
ENSG00000000938    4175.932584
ENSG00000000971      24.930681
ENSG00000001036     146.164725
ENSG00000001084     113.047477
ENSG00000001167     198.247178
Name: tpm_count, dtype: float64

2020_UGA — average tpm_count per gene (first 10):
ensembl_gene_id
ENSG00000000003      0.236586
ENSG00000000005      0.000135
ENSG00000000419     46.332111
ENSG00000000457      3.285624
ENSG00000000460      1.235722
ENSG00000000938    231.668096
ENSG00000000971      1.825762
ENSG00000001036     25.082266
ENSG00000001084      6.389380
ENSG00000001167     13.065100
Name: tpm_count, dtype: float64

SDY2867 — average tpm_count per gene (first 10):
ensembl_gene_id
ENSG00000000003      0.217756
ENSG00000000005      0.000000
ENSG00000000419     38.816097
ENSG00000000457      8.268567
ENSG00

In [6]:
genes_2019 = set(eda_2019['ensembl_gene_id'].unique())
genes_2020 = set(eda_2020['ensembl_gene_id'].unique())
genes_sdy  = set(eda_sdy['ensembl_gene_id'].unique())

common_genes = genes_2019 & genes_2020 & genes_sdy
print(f"Genes per study — 2019: {len(genes_2019)}, 2020: {len(genes_2020)}, SDY2867: {len(genes_sdy)}")
print(f"Common genes across all three: {len(common_genes)}\n")

for name, df in [('2019_UGA', eda_2019), ('2020_UGA', eda_2020), ('SDY2867', eda_sdy)]:
    mean = df[df['ensembl_gene_id'].isin(common_genes)].groupby('ensembl_gene_id')['tpm_count'].mean()
    print(f"{name} — mean tpm_count over common genes (first 10):")
    print(mean.head(10))
    print()

Genes per study — 2019: 42531, 2020: 60617, SDY2867: 54902
Common genes across all three: 32902

2019_UGA — mean tpm_count over common genes (first 10):
ensembl_gene_id
ENSG00000000003       3.273456
ENSG00000000005       0.001650
ENSG00000000419     100.951028
ENSG00000000457     148.218532
ENSG00000000460      24.120888
ENSG00000000938    4175.932584
ENSG00000000971      24.930681
ENSG00000001036     146.164725
ENSG00000001084     113.047477
ENSG00000001167     198.247178
Name: tpm_count, dtype: float64

2020_UGA — mean tpm_count over common genes (first 10):
ensembl_gene_id
ENSG00000000003      0.236586
ENSG00000000005      0.000135
ENSG00000000419     46.332111
ENSG00000000457      3.285624
ENSG00000000460      1.235722
ENSG00000000938    231.668096
ENSG00000000971      1.825762
ENSG00000001036     25.082266
ENSG00000001084      6.389380
ENSG00000001167     13.065100
Name: tpm_count, dtype: float64

SDY2867 — mean tpm_count over common genes (first 10):
ensembl_gene_id
ENSG00000000

In [7]:
def clean_transcriptomics(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH + '/' + filename, sep='\t')
    df = df.drop(columns=['transcriptomics_id', 'raw_count', 'material'])

    df = df[df['timepoint'].isin([0, 7])]
    df = df[df['ensembl_gene_id'].isin(unique_genes_challenge)]

    df_pivot = df.pivot_table(
        index='participant_id',
        columns=['timepoint', 'ensembl_gene_id'],
        values='tpm_count'
    )
    df_pivot.columns = [f'TRAN_{gene}_d{int(tp)}' for tp, gene in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    df_pivot = log_standard_scale(df_pivot)
    return df_pivot

## Process each dataset

In [8]:
df_2019 = clean_transcriptomics('train_transcriptomics_2019_UGA.tsv')
df_2019.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_2019_UGA_cleaned.parquet', index=False)
print(f"2019_UGA: {df_2019.shape}")
peek(df_2019)

KeyboardInterrupt: 

In [ ]:
df_2020 = clean_transcriptomics('train_transcriptomics_2020_UGA.tsv')
df_2020.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_2020_UGA_cleaned.parquet', index=False)
print(f"2020_UGA: {df_2020.shape}")
peek(df_2020)

In [ ]:
df_sdy = clean_transcriptomics('train_transcriptomics_SDY2867.tsv')
df_sdy.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_SDY2867_cleaned.parquet', index=False)
print(f"SDY2867: {df_sdy.shape}")
peek(df_sdy)

## Combine and save

In [ ]:
df_combined = pd.concat([df_2019, df_2020, df_sdy], axis=0).reset_index(drop=True)
df_combined.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_combined_cleaned.parquet', index=False)
print(f"Combined: {df_combined.shape}")
peek(df_combined)